In [22]:
# %%
# !pip install youtube-transcript-api scrapetube

from youtube_transcript_api import YouTubeTranscriptApi
import scrapetube
from urllib.parse import urlparse, parse_qs
import os

In [23]:
# %%
# --- Pipeline Configuration ---
# Feed the queue channel handles or URLs instead of hardcoded video links.

experts_queue = {
    "josh_braun": "",
    "guillaume_moubeche": "",
    "jason_bay": "https://www.youtube.com/@jasondbay", 
    "alex_berman": "",
    "jeremy_chatelaine": "",
    "john_barrows": "",
    "nick_abraham": "",
    "becc_holland": "",
    "will_allred": "",
    "jed_mahrle": ""
}

# Number of recent videos to pull per expert
MAX_VIDEOS_PER_EXPERT = 2

OUTPUT_DIR = "../research/youtube-transcripts"

In [24]:
# %%
def extract_video_id(url: str) -> str:
    """Robust URL parser for all YouTube link formats."""
    parsed = urlparse(url)
    if parsed.hostname == 'youtu.be':
        return parsed.path[1:]
    if parsed.hostname in ('www.youtube.com', 'youtube.com'):
        if parsed.path == '/watch':
            return parse_qs(parsed.query).get('v', [None])[0]
    return url

In [25]:
# %%
# Dynamic Extraction Layer

transcripts_data = []
ytt_api = YouTubeTranscriptApi()

for expert, channel_url in experts_queue.items():
    if not channel_url:
        continue # Skip empty entries
        
    print(f"\nScanning channel for {expert}...")
    
    try:
        # 1. Fetch the latest video IDs from the channel
        videos = scrapetube.get_channel(channel_url=channel_url, limit=MAX_VIDEOS_PER_EXPERT)
        video_ids = [vid['videoId'] for vid in videos]
        
        print(f"  Found {len(video_ids)} recent videos. Extracting transcripts...")
        
        # 2. Extract transcripts for those specific videos
        for vid_id in video_ids:
            try:
                raw_transcript = ytt_api.fetch(vid_id)
                full_text = " ".join([segment.text for segment in raw_transcript])
                clean_text = full_text.replace(". ", ".\n\n")
                
                video_url = f"https://www.youtube.com/watch?v={vid_id}"
                
                transcripts_data.append({
                    "expert": expert,
                    "video_id": vid_id,
                    "url": video_url,
                    "content": clean_text
                })
                print(f"    ✓ Success: [{vid_id}]")
                
            except Exception as e:
                # Sometimes videos don't have captions enabled, we just catch and skip
                print(f"    ✗ Failed on [{vid_id}]: {e}")
                
    except Exception as e:
        print(f"  ✗ Failed to scan channel: {e}")

print("\nDynamic Data Extraction Complete.")


Scanning channel for jason_bay...
  Found 2 recent videos. Extracting transcripts...
    ✓ Success: [WvVNB-2ylSg]
    ✓ Success: [cQZ15-bivvs]

Dynamic Data Extraction Complete.


In [26]:
# %%
import glob

# Ensure the target directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# THE UPGRADE: Clear out old markdown files before saving the new batch
print("Sweeping the directory for old data...")
old_files = glob.glob(os.path.join(OUTPUT_DIR, "*.md"))
for f in old_files:
    os.remove(f)

for data in transcripts_data:
    # Flat file naming convention: expert_videoid.md
    filename = os.path.join(OUTPUT_DIR, f"{data['expert']}_{data['video_id']}.md")
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Transcript: {data['expert'].replace('_', ' ').title()}\n\n")
        f.write(f"**Video ID:** {data['video_id']}\n")
        f.write(f"**Source URL:** {data['url']}\n\n")
        f.write("---\n\n")
        f.write("### Content\n\n")
        f.write(data['content'])

print("Files saved to storage. Performing directory audit...\n")

# QA check to verify file sizes
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"  {file} — ({size} bytes)")

Sweeping the directory for old data...
Files saved to storage. Performing directory audit...

  jason_bay_cQZ15-bivvs.md — (756 bytes)
  jason_bay_WvVNB-2ylSg.md — (937 bytes)
